In [34]:
import nibabel as nib
import nibabel.orientations as nio

files= ['/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30020_ses-d0092_run-02_T1w.nii.gz','/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30032_ses-d0262_run-02_T1w.nii.gz','/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30048_ses-d0983_run-02_T1w.nii.gz','/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30059_ses-d0230_run-02_T1w.nii.gz','/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30062_ses-d0087_run-02_T1w.nii.gz','/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30073_ses-d0033_run-02_T1w.nii.gz']

for file_path in files:
    img= nib.load(file_path)
    img_data=img.get_fdata()
    # print(img_data.shape)
    print(img.header)
    
    affine = img.affine
    # print("\nAffine matrix:\n", affine)
    
    # Orientation of each axis
    ornt = nio.io_orientation(affine)
    axcodes = nio.ornt2axcodes(ornt)
    
    print("\nOrientation of axes (i, j, k):")
    print("i-axis:", axcodes[0])
    print("j-axis:", axcodes[1])
    print("k-axis:", axcodes[2])

    break

<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : np.bytes_(b'')
db_name         : np.bytes_(b'?TR:0.000 TE:0.00')
extents         : 0
session_error   : 0
regular         : np.bytes_(b'r')
dim_info        : 0
dim             : [  3 256 256 128   1   1   1   1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : int16
bitpix          : 16
slice_start     : 0
pixdim          : [1.   1.   1.   1.25 0.   1.   1.   0.  ]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffset         : 0.0
glmax           : 255
glmin           : 0
descrip         : np.bytes_(b'removed')
aux_file        : np.bytes_(b'OAS30020_MR_d0092')
qform_code      : unknown
sform_code      : unknown
quatern_b       : 0.0
quatern_c       : 0.0
quatern_d       : 0.0
qoffset

## Note

Although the orientation is reported as LAS, this does not imply that the image is anatomically oriented in LAS space. The NIfTI header contains neither a valid qform nor sform, and the affine matrix is the identity. In the absence of physical orientation information, nibabel returns a default orientation purely based on the assumed voxel axis directions. Consequently, the reported LAS orientation is a heuristic fallback and should not be interpreted as the true anatomical orientation of the MRI volume.

In [67]:
# orientation of correct file

normal_file = '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30049_ses-d0013_run-01_T1w.nii.gz'
img= nib.load(normal_file)
img_data=img.get_fdata()
print(img_data.shape)
# print(img.header)

affine = img.affine
# print("\nAffine matrix:\n", affine)

# Orientation of each axis
ornt = nio.io_orientation(affine)
axcodes = nio.ornt2axcodes(ornt)

print("\nOrientation of axes (i, j, k):")
print("i-axis:", axcodes[0])
print("j-axis:", axcodes[1])
print("k-axis:", axcodes[2])

(176, 256, 256)

Orientation of axes (i, j, k):
i-axis: R
j-axis: A
k-axis: S


In [33]:
# if we use  nib.as_closest_canonical for reorient

file_path= '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30020_ses-d0092_run-02_T1w.nii.gz'
img = nib.load(file_path)

# Convert to closest RAS orientation
img_ras = nib.as_closest_canonical(img)

# Verify
affine = img_ras.affine
ornt = nio.io_orientation(affine)
axcodes = nio.ornt2axcodes(ornt)

print("After conversion:")
print("i-axis:", axcodes[0])
print("j-axis:", axcodes[1])
print("k-axis:", axcodes[2])

# Save
out_path = '/home/jupyter-nafisha/Brain_age/sub-OAS30020_ses-d0092_run-02_T1w_RAS.nii.gz'
nib.save(img_ras, out_path)

After conversion:
i-axis: R
j-axis: A
k-axis: S


## Note:-

nib.as_closest_canonical relies entirely on the orientation encoded in the NIfTI affine. In this case, because the header contains no valid qform or sform, the affine is the identity and the reported LAS orientation is a default assumption rather than a true anatomical orientation. Consequently, as_closest_canonical reorients the image based on this assumed LAS orientation. While the output is labeled as RAS, there is no guarantee that it corresponds to the true anatomical RAS orientation of the MRI volume.

In [ ]:
# for all orientations

import os
import numpy as np
import nibabel as nib
import itertools
from nibabel.orientations import apply_orientation, io_orientation

def all_48_orientations():
    """
    Generate all 48 possible voxel-space 3D orientations.
    Each orientation is a (3,2) array suitable for nibabel.apply_orientation.
    """
    orientations = []
    for perm in itertools.permutations([0, 1, 2]):      # 6 axis permutations
        for signs in itertools.product([1, -1], repeat=3):  # 8 sign combinations
            ornt = np.array([[perm[i], signs[i]] for i in range(3)])
            orientations.append(ornt)
    return orientations

def orientation_label(orient):
    """
    Convert a nibabel orientation matrix to a string like 'RAS', 'LPS', etc.
    """
    axis_labels = ['R', 'A', 'S']
    neg_labels = ['L', 'P', 'I']
    label = ''
    for ax, direction in orient:
        if direction == 1:
            label += axis_labels[ax]
        else:
            label += neg_labels[ax]
    return label

def save_all_orientations_with_labels(image_path, folder_path):
    # Load original image
    img = nib.load(image_path)
    data = img.get_fdata()
    header = img.header.copy()
    
    # Create output folder if it doesn't exist
    os.makedirs(folder_path, exist_ok=True)
    
    # Get all 48 orientations
    orientations = all_48_orientations()
    
    for idx, ornt in enumerate(orientations):
        # Apply orientation to voxel data
        new_data = apply_orientation(data, ornt)
        
        # Get the orientation label
        label = orientation_label(ornt)
        
        # Create a new affine: just keep voxel sizes, origin=0
        zooms = header.get_zooms()[:3]
        affine = np.diag(list(zooms) + [1])
        
        # Save new NIfTI
        new_img = nib.Nifti1Image(new_data, affine, header=header)
        out_path = os.path.join(folder_path, f"image_{label}.nii.gz")
        nib.save(new_img, out_path)
        print(f"Saved: {out_path}")


# Example usage:
image_path = '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30032_ses-d0262_run-02_T1w.nii.gz'
folder_path = "/home/jupyter-nafisha/Brain_age/oriented_files"

save_all_orientations_with_labels(image_path, folder_path)

## Result
As brain is symmetrical along y axis so there can be 2 correct orientations (from all 48 orientations).
all the 48 orientations corresponding to single file are visualized and it was found out that PSR and PSL are two orientations which are correct

In [77]:
# For PSR and PSL orientation only 

import os
import numpy as np
import nibabel as nib
from nibabel.orientations import apply_orientation

def save_psr_psl(image_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    base_name = os.path.basename(image_path).replace(".nii.gz", "").replace(".nii", "")

    img = nib.load(image_path)
    data = img.get_fdata()
    header = img.header.copy()

    zooms = header.get_zooms()[:3]

    # Define orientations
    PSR = np.array([[1, -1],
                    [2,  1],
                    [0,  1]])

    PSL = np.array([[1, -1],
                    [2,  1],
                    [0, -1]])

    orientations = {
        "PSR": PSR,
        "PSL": PSL
    }

    for label, ornt in orientations.items():
        new_data = apply_orientation(data, ornt)

        # Minimal affine encoding voxel sizes
        affine = np.diag(list(zooms) + [1])

        new_img = nib.Nifti1Image(new_data, affine, header=header)
        out_path = os.path.join(out_dir, base_name + f"_{label}.nii.gz")
        nib.save(new_img, out_path)

        print(f"Saved {label}: {out_path}")

# Example usage

# image_path = '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30048_ses-d0983_run-02_T1w.nii.gz'
# image_path = '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30059_ses-d0230_run-02_T1w.nii.gz'
# image_path = '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30062_ses-d0087_run-02_T1w.nii.gz'
image_path = '/home/jupyter-nafisha/Brain_age/OASIS3/sub-OAS30073_ses-d0033_run-02_T1w.nii.gz'

out_dir = "/home/jupyter-nafisha/Brain_age/oriented_files"

save_psr_psl(image_path, out_dir)

Saved PSR: /home/jupyter-nafisha/Brain_age/oriented_files/sub-OAS30073_ses-d0033_run-02_T1w_PSR.nii.gz
Saved PSL: /home/jupyter-nafisha/Brain_age/oriented_files/sub-OAS30073_ses-d0033_run-02_T1w_PSL.nii.gz


In [78]:
import subprocess

model_path = "/home/jupyter-nafisha/Brain_age/synthstrip.pt"
strip_out = "/home/jupyter-nafisha/Brain_age/oriented_files/sub-OAS30073_ses-d0033_run-02_T1w_PSL_stripped.nii.gz"
input_path= "/home/jupyter-nafisha/Brain_age/oriented_files/sub-OAS30073_ses-d0033_run-02_T1w_PSL.nii.gz"

cmd = [
    "/home/jupyter-nafisha/.local/bin/nipreps-synthstrip",
    "-i", input_path,
    "-o", strip_out,
    "--model", model_path
]
result = subprocess.run(cmd, capture_output=True, text=True)

In [79]:
ants_bin = "/home/jupyter-nafisha/Brain_age/ants_2_6_3/ants-2.6.3/bin"
env = os.environ.copy()
env["PATH"] = ants_bin + ":" + env["PATH"]
template_path = '/home/jupyter-nafisha/Brain_age/templates/MNI152_T1_1mm_Brain.nii.gz'

base_name = input_path.replace(".nii.gz", "").replace(".nii", "")
output_path =  base_name +'_'
reg_cmd = [
    f"{ants_bin}/antsRegistrationSyNQuick.sh",
    "-d", "3",
    "-f", template_path,
    "-m", strip_out,
    "-o", output_path,
    "-n", str(os.cpu_count()),   
    "-t", "a"
]
subprocess.run(reg_cmd, env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# output_path

CompletedProcess(args=['/home/jupyter-nafisha/Brain_age/ants_2_6_3/ants-2.6.3/bin/antsRegistrationSyNQuick.sh', '-d', '3', '-f', '/home/jupyter-nafisha/Brain_age/templates/MNI152_T1_1mm_Brain.nii.gz', '-m', '/home/jupyter-nafisha/Brain_age/oriented_files/sub-OAS30073_ses-d0033_run-02_T1w_PSL_stripped.nii.gz', '-o', '/home/jupyter-nafisha/Brain_age/oriented_files/sub-OAS30073_ses-d0033_run-02_T1w_PSL_', '-n', '32', '-t', 'a'], returncode=0)